# Building a Chatbot with Memory using LangChain

In [1]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

### Solution taken from the Course itself:
I'll try creating my own chatbot later on.

In [3]:
from langchain.memory import ConversationBufferMemory, ChatMessageHistory
from langchain.chains import ConversationChain
from langchain_core.messages import HumanMessage, AIMessage

# 1. Set up the language model
model_id = 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8'
parameters = {
    GenParams.MAX_NEW_TOKENS: 256,
    GenParams.TEMPERATURE: 0.2,
}
credentials = {"url": "https://us-south.ml.cloud.ibm.com"}
project_id = "skills-network"

# Initialize the model
model = ModelInference(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)
llm = WatsonxLLM(model=model)

# 2. Create a simple conversation with chat history
history = ChatMessageHistory()

# Add some initial messages
history.add_user_message("Hello, my name is Alice.")
history.add_ai_message("Hello Alice! It's nice to meet you. How can I help you today?")

# 3. Print the current conversation history
print("Initial Chat History:")
for message in history.messages:
    sender = "Human" if isinstance(message, HumanMessage) else "AI"
    print(f"{sender}: {message.content}")

# 4. Set up a conversation chain with memory
memory = ConversationBufferMemory(chat_memory=history)
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

# 5. Function to simulate a conversation
def chat_simulation(conversation, inputs):
    """Run a series of inputs through the conversation chain and display responses"""
    print("\n=== Beginning Chat Simulation ===")
    
    for i, user_input in enumerate(inputs):
        print(f"\n--- Turn {i+1} ---")
        print(f"Human: {user_input}")
        
        # Get response from the conversation chain
        response = conversation.invoke(input=user_input)
        
        # Print the AI's response
        print(f"AI: {response['response']}")
    
    print("\n=== End of Chat Simulation ===")

# 6. Test with a series of related questions
test_inputs = [
    "My favorite color is blue.",
    "I enjoy hiking in the mountains.",
    "What activities would you recommend for me?",
    "What was my favorite color again?",
    "Can you remember both my name and my favorite color?"
]

chat_simulation(conversation, test_inputs)

# 7. Examine the conversation memory
print("\nFinal Memory Contents:")
print(conversation.memory.buffer)

# 8. Create a new conversation with a different type of memory (optional)
from langchain.memory import ConversationSummaryMemory

# Create a summarizing memory that will compress the conversation
summary_memory = ConversationSummaryMemory(llm=llm)
# Save the initial context to the summary memory
summary_memory.save_context(
    {"input": "Hello, my name is Alice."}, 
    {"output": "Hello Alice! It's nice to meet you. How can I help you today?"}
)
summary_conversation = ConversationChain(
   llm=llm,
   memory=summary_memory,
   verbose=True
)

print("\\\\\\\\\\\\n\\n=== Testing Conversation Summary Memory ===")
# Let's use the same inputs for comparison
chat_simulation(summary_conversation, test_inputs)

print("\\nFinal Summary Memory Contents:")
print(summary_memory.buffer)

# 9. Compare the two memory types
print("\n=== Memory Comparison ===")
print(f"Buffer Memory Size: {len(conversation.memory.buffer)} characters")
print(f"Summary Memory Size: {len(summary_memory.buffer)} characters")
print("\nThe conversation summary memory typically creates a more compact representation of the chat history.")

Initial Chat History:
Human: Hello, my name is Alice.
AI: Hello Alice! It's nice to meet you. How can I help you today?

=== Beginning Chat Simulation ===

--- Turn 1 ---
Human: My favorite color is blue.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hello, my name is Alice.
AI: Hello Alice! It's nice to meet you. How can I help you today?
Human: My favorite color is blue.
AI:

> Finished chain.
AI:  That's a great choice! Blue is a very calming color. Did you know that the Pantone Color Institute has designated several shades of blue as "Color of the Year" over the years? For example, in 2015, they chose "Marsala", a reddish-brown color, but in 2016, they chose "Rose Quartz" and "Serenity", with "Serenity"

### Explaination:
(From point 5 onwards)